# Put the Moss model in your own Hugging Face account

This optional notebook copies the already-converted model used by Moss. It does not quantize again, train, or generate images. No GPU is needed. The app works with its public default download without doing this.

1. Use a normal CPU Colab session. Choose **Runtime → Run all**.
2. Paste your Hugging Face **Write** token into the hidden input when asked. Do not paste it into a code cell.
3. Leave this tab open. About **13.3 GB** is downloaded and uploaded. Free Colab can disconnect; run again if that happens.
4. Copy the final **manifest URL** into Moss → Settings → Model options → Optional mirror manifest URL → Use mirror.

The notebook creates a **public** model repository named `Moss-iOS-Model` in your account. Only the public model files and attribution are uploaded. Your token is not written to that repository. Revoke the token afterward if you no longer need it.


In [ ]:
%pip -q install "huggingface_hub>=0.34,<2" hf_xet


In [ ]:
from pathlib import Path
import getpass, hashlib, json, os, shutil, urllib.parse
from huggingface_hub import HfApi, hf_hub_download
from IPython.display import display, Markdown

manifest = {'format': 1, 'identity': 'moss-base8-text6-v1', 'name': 'MOSS Base · 8-bit image model / 6-bit text encoder', 'files': [{'path': 'tokenizer/chat_template.jinja', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/tokenizer/chat_template.jinja', 'bytes': 7756, 'sha256': 'a4aee8afcf2e0711942cf848899be66016f8d14a889ff9ede07bca099c28f715', 'component': 'tokenizer'}, {'path': 'tokenizer/tokenizer.json', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/tokenizer/tokenizer.json', 'bytes': 19989325, 'sha256': '06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523', 'component': 'tokenizer'}, {'path': 'tokenizer/tokenizer_config.json', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/tokenizer/tokenizer_config.json', 'bytes': 1123, 'sha256': 'bee8eba30f0eb4af73c0fe2cd06d0f89b657d7819941c438157ec42f7c80ea87', 'component': 'tokenizer'}, {'path': 'transformer/config.json', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/transformer/config.json', 'bytes': 352, 'sha256': 'e0395aaa8bf5902475a7fb3b6a034e27c13b3a733e36d8608eeb4d98c539abc1', 'component': 'transformer'}, {'path': 'transformer/model-00001-of-00002.safetensors', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/transformer/model-00001-of-00002.safetensors', 'bytes': 4291488455, 'sha256': 'eeefad1f9ea90e52f0d84ba9d867044ab2afae05bc878a056dd36633dac6bebd', 'component': 'transformer'}, {'path': 'transformer/model-00002-of-00002.safetensors', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/transformer/model-00002-of-00002.safetensors', 'bytes': 1201822635, 'sha256': 'a4fdc714be66a351c5306095835b2c6f8309559c1c79129bdc457e79e4f5628e', 'component': 'transformer'}, {'path': 'transformer/model.safetensors.index.json', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/transformer/model.safetensors.index.json', 'bytes': 66602, 'sha256': 'd497fcdb25d8e037989754eee16773fd3b0a33987f954a50c835b2976e3fe48a', 'component': 'transformer'}, {'path': 'vae/config.json', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/vae/config.json', 'bytes': 935, 'sha256': '0e93e95bcc04f975f8d6a9ac5308d5a3587c79ce4ea3dfe38ef6d5bf7cfbcce8', 'component': 'vae'}, {'path': 'vae/decoder.safetensors', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-Base-MLX-8bit/resolve/f06abf870b456dd7a57198473a445ccb6661c933/vae/decoder.safetensors', 'bytes': 111805440, 'sha256': 'd6bbd213504840e9f29a280aacb42ba5bb5de9e3ec0a1ca4f7f66302e2c26ad5', 'component': 'vae'}, {'path': 'text_encoder/config.json', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-SFT-MLX-compact/resolve/5b0434d1635431db9eee01c0918a246c745300ba/text_encoder/config.json', 'bytes': 2474, 'sha256': '1d28810329ccc2017967d38abd30fc6c2ae446f524c9a80e9c977f9cff67b10f', 'component': 'text'}, {'path': 'text_encoder/model-00001-of-00002.safetensors', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-SFT-MLX-compact/resolve/5b0434d1635431db9eee01c0918a246c745300ba/text_encoder/model-00001-of-00002.safetensors', 'bytes': 4259639915, 'sha256': 'f9c1ecd81f19ca0ba1fd531ced7269f2cfb853b74160b18218d11d24ad70661c', 'component': 'text'}, {'path': 'text_encoder/model-00002-of-00002.safetensors', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-SFT-MLX-compact/resolve/5b0434d1635431db9eee01c0918a246c745300ba/text_encoder/model-00002-of-00002.safetensors', 'bytes': 2190818161, 'sha256': '43260ac0a16b2324ebb60e931ce8831e14c38a3d0b5eb09b8fa9881fe4ba82cc', 'component': 'text'}, {'path': 'text_encoder/model.safetensors.index.json', 'url': 'https://huggingface.co/VisualInference/MOSS-Image-SFT-MLX-compact/resolve/5b0434d1635431db9eee01c0918a246c745300ba/text_encoder/model.safetensors.index.json', 'bytes': 69862, 'sha256': 'f091341d243bfb276250e9e3daba34af0de9ec13022d14c9e16b2c1ed5329f47', 'component': 'text'}, {'path': 'adapter/lora.safetensors', 'url': 'https://huggingface.co/lodestones/MOSS-Image-Base-8-steps-dmd/resolve/da05cb3b52899179e2bb5f7efe1058fefcbae21b/lora.safetensors', 'bytes': 1189176800, 'sha256': 'f0e6cd07f222dfaea798afe3b796e2c2995cadfd44ad56e02ca616f0467aec28', 'component': 'adapter'}]}
work = Path("/content/moss-mirror")
export = work / "export"
export.mkdir(parents=True, exist_ok=True)
token = getpass.getpass("Paste your Hugging Face WRITE token here (it stays hidden): ").strip()
if not token:
    raise ValueError("No token entered. Run this cell again and paste a Write token.")
api = HfApi(token=token)
username = api.whoami()["name"]
repo_id = username + "/Moss-iOS-Model"
api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)
print("Copying the supported public model to https://huggingface.co/" + repo_id)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for index, entry in enumerate(manifest["files"], 1):
    parts = urllib.parse.urlparse(entry["url"]).path.strip("/").split("/")
    source_repo = "/".join(parts[:2])
    revision = parts[3]
    filename = "/".join(parts[4:])
    print(f"[{index}/{len(manifest['files'])}] Downloading {entry['path']}")
    path = Path(hf_hub_download(repo_id=source_repo, filename=filename, revision=revision,
                              local_dir=work / "sources" / source_repo.replace("/", "--"), token=token))
    if path.stat().st_size != entry["bytes"] or sha256(path) != entry["sha256"]:
        raise RuntimeError("Integrity check failed for " + entry["path"] + ". Run all again to retry.")
    destination = export / entry["path"]
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    try:
        os.link(path, destination)
    except OSError:
        shutil.copy2(path, destination)

readme = """---
license: apache-2.0
library_name: mlx
pipeline_tag: text-to-image
base_model: OpenMOSS-Team/MOSS-Image-Base
---
# Moss iOS model mirror
A byte-for-byte mirror for the experimental Moss iOS app.

Image transformer: VisualInference/MOSS-Image-Base-MLX-8bit (8-bit).
Text encoder: VisualInference/MOSS-Image-SFT-MLX-compact (6-bit, shared Qwen encoder).
Decoder: FP32 FLUX.2 small decoder from the Base MLX export.
Fast adapter: lodestones/MOSS-Image-Base-8-steps-dmd, unmerged FP32.

Original model: https://huggingface.co/OpenMOSS-Team/MOSS-Image-Base
MLX conversion: https://github.com/felix-ab/moss-image-mlx
Model and derivative conversion licensing: Apache-2.0; see LICENSE and NOTICE.
This mirror is not an official OpenMOSS release and does not establish iPhone compatibility.
"""
(export / "README.md").write_text(readme)
import urllib.request
license_text = urllib.request.urlopen("https://www.apache.org/licenses/LICENSE-2.0.txt").read()
(export / "LICENSE").write_bytes(license_text)
(export / "NOTICE").write_text("MOSS-Image: OpenMOSS contributors. MLX conversion: Felix Abramovich and contributors, 2026. DMD adapter: lodestones contributors. Original source and pinned weight revisions are recorded in model-manifest.json. This is an unmodified weight mirror.\n")
print("Uploading verified weights. This can take a long time; keep this tab open.")
commit = api.upload_folder(repo_id=repo_id, repo_type="model", folder_path=str(export),
                          commit_message="Mirror pinned Moss iOS model components")
revision = commit.oid
for entry in manifest["files"]:
    entry["url"] = f"https://huggingface.co/{repo_id}/resolve/{revision}/{entry['path']}"
manifest_path = work / "model-manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
api.upload_file(path_or_fileobj=str(manifest_path), path_in_repo="model-manifest.json",
                repo_id=repo_id, repo_type="model", commit_message="Publish verified Moss iOS download manifest")
url = f"https://huggingface.co/{repo_id}/resolve/main/model-manifest.json"
print("FINISHED. Copy this complete address into Moss:")
print(url)
display(Markdown(f"[Open your model files](https://huggingface.co/{repo_id}/tree/main) · [Open your manifest]({url})"))
del token
